# 在 Notebook 中运行 DR-Learner 流水线

本目录：`dr_learner_pipeline/notebook/`。流水线入口为 **`python -m dr_learner_pipeline.run_pipeline`**，**工作目录必须是 `model_build` 根**（与 `utils`、`dr_learner_pipeline` 同级），否则模块路径会失败。

**CLI 与 Notebook 对照**：命令行可传 **`--db-config-json`**；Notebook 中可用 **`run_pipeline_notebook(..., db_config=DB_CONFIG)`**（或 **`db_config_json_path=`**）。

**两步（推荐调试）**：先 **`load_wide_and_split`** 得到 **`bundle`**（可检查 `bundle.df` / `bundle.train_df` 等），再 **`run_pipeline_training_from_splits(..., bundle, verbose=True)`** 看 `[DR pipeline]` 进度行。一键且要带进度时：`run_pipeline_notebook(..., progress_prints=True)`。

**Notebook 参数**：下面 **参数总表** 一节提供内存 **`SPLIT_DATES`**、**`PIPELINE_OVERRIDES`** 与 **`CONFIG_YAML`**（相对 `MODEL_BUILD`）；CLI 仍用 `--split_dates_path` 文件。

## 数据与配置

- **MySQL**：连接信息只从 JSON 读取（见 `utils/db_config.py`）。默认文件为 **`feature_engineering/db_config.json`**；也可用环境变量 **`LIFT_DB_CONFIG_JSON`** 指向任意路径（应在 **import 任何 `utils.*` 之前** 设置该变量）。
- **宽表**：须在 YAML 中配置 `db_table`，或通过命令行 **`--db-table`** 覆盖；若使用 **`wide_table_path`** 读 parquet，则不需要数据库表名。
- **勿将** `db_config.json`、GitHub token 等密钥文件提交 Git。

## 依赖

在 `model_build` 下：`pip install -r requirements_dr_pipeline.txt`（见文末单元格）。

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def resolve_model_build() -> Path:
    """Locate model_build root (contains utils/ and dr_learner_pipeline/)."""
    marker = Path("utils") / "wide_table_db.py"
    for start in (Path.cwd(), Path.cwd() / "model_build"):
        p = start.resolve()
        for _ in range(8):
            if (p / marker).is_file():
                return p
            if p == p.parent:
                break
            p = p.parent
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        nb_path = Path(str(nb)).resolve()
        cand = nb_path.parent.parent.parent
        if (cand / marker).is_file():
            return cand
    raise RuntimeError(
        "Could not find model_build. cd to model_build or set MODEL_BUILD_OVERRIDE below."
    )


MODEL_BUILD_OVERRIDE: Path | None = None  # e.g. Path("/path/to/feature_engineering/model_build")

MODEL_BUILD = MODEL_BUILD_OVERRIDE or resolve_model_build()
os.chdir(MODEL_BUILD)
if str(MODEL_BUILD) not in sys.path:
    sys.path.insert(0, str(MODEL_BUILD))

print("MODEL_BUILD:", MODEL_BUILD.resolve())
print("cwd:", Path.cwd())

MODEL_BUILD: /Users/jialuliang/Documents/WorkMagic/lift_test_calibration/feature_engineering/model_build
cwd: /Users/jialuliang/Documents/WorkMagic/lift_test_calibration/feature_engineering/model_build


In [2]:
# 可选：在 import utils 之前设置自定义 DB JSON 路径
# os.environ["LIFT_DB_CONFIG_JSON"] = str(MODEL_BUILD.parent / "db_config.json")

from utils.db_config import resolve_db_config_json_path

db_json = resolve_db_config_json_path()
print("DB config JSON path:", db_json)
print("exists:", db_json.is_file())

DB config JSON path: /Users/jialuliang/Documents/WorkMagic/lift_test_calibration/feature_engineering/db_config.json
exists: True


## 参数总表（内存 `SPLIT_DATES` + YAML 基底 + `PIPELINE_OVERRIDES`）

下一格定义 **`CONFIG_YAML`**（相对 **`MODEL_BUILD`**）、**`SPLIT_DATES`**、**`PIPELINE_OVERRIDES`**、**`DB_TABLE`**。无需单独的 `split_dates.json`；训练时会在 `exp_dir` 写入 **`notebook_split_dates.json`**。

**CLI** 仍使用 `--split_dates_path` 指向 JSON 文件。


In [ ]:
CONFIG_YAML = "dr_learner_pipeline/config/pipeline_grid.yaml"

SPLIT_DATES = {
    "train": [
        "2025-07-01",
        "2025-07-15",
        "2025-08-01",
        "2025-08-15",
        "2025-09-01",
        "2025-09-15",
        "2025-10-01",
        "2025-10-15",
        "2025-11-01",
        "2025-11-15",
        "2025-11-28",
    ],
    "val": ["2025-12-01", "2025-12-15"],
    "test": ["2025-12-25", "2026-01-01"],
}

PIPELINE_OVERRIDES: dict = {}

DB_TABLE: str | None = None  # 覆盖 YAML，例如 "workmagic_bi.your_wide_table"


## 运行流水线（进程内两步）

1. **加载 + 划分**：下面第一格用 `merge_pipeline_config` 与内存 **`SPLIT_DATES`** 调用 `load_wide_and_split`，得到 `bundle`。
2. **训练**：第二格对同一 **`SPLIT_DATES`** 调用 `run_pipeline_training_from_splits`。

需要覆盖表名时使用上一节的 **`DB_TABLE`** 或在 `cfg` 上设置 `db_table`。


In [ ]:
from dr_learner_pipeline.run_pipeline import (
    _load_yaml,
    load_wide_and_split,
    merge_pipeline_config,
)

CONFIG_PATH = (MODEL_BUILD / CONFIG_YAML).resolve()
cfg = merge_pipeline_config(_load_yaml(CONFIG_PATH), PIPELINE_OVERRIDES)
if DB_TABLE:
    cfg["db_table"] = DB_TABLE.strip()
# elif need table: cfg["db_table"] = "workmagic_bi.your_wide_table"

bundle = load_wide_and_split(cfg, SPLIT_DATES)
print(
    "df shape:", bundle.df.shape,
    "| load window:", bundle.date_load_start, "..", bundle.date_load_end,
)
print("train", len(bundle.train_df), "val", len(bundle.val_df), "test", len(bundle.test_df))


### 训练阶段

使用上一格的 **`cfg`** 与 **`bundle`**（勿在中间改列结构）。`verbose=True` 会在 stdout 打印进度。

In [ ]:
from dr_learner_pipeline.run_pipeline import run_pipeline_training_from_splits

cli_overrides: dict = {}
if DB_TABLE:
    cli_overrides["db_table"] = DB_TABLE.strip()
elif cfg.get("db_table"):
    cli_overrides["db_table"] = str(cfg["db_table"]).strip()

exp_dir = run_pipeline_training_from_splits(
    cfg,
    SPLIT_DATES,
    CONFIG_PATH,
    cli_overrides,
    bundle,
    verbose=True,
)
print("Done:", exp_dir)


## 等价 shell

```bash
cd /path/to/feature_engineering/model_build
python -m dr_learner_pipeline.run_pipeline \\
  --config dr_learner_pipeline/config/pipeline_grid.yaml \\
  --split_dates_path dr_learner_pipeline/example_split_dates.json \\
  --db-table workmagic_bi.your_wide_table
```

产物在 `model_build/output/<时间戳>/`。解析结果见 **`parse_experiment_results.ipynb`**。

## 安装依赖（按需）

In [ ]:
# !{sys.executable} -m pip install -q -r requirements_dr_pipeline.txt